# Carga de datos - Modelo de riesgo crediticio (PI M5)

**Contexto de negocio.** Somos el equipo de Datos y Analitica de una entidad financiera. Cada credito otorgado
queda registrado con informacion del solicitante (edad, tipo laboral, salario), del producto (capital, plazo, cuota)
y de su historial en la central de riesgo (puntaje Datacredito, creditos vigentes, saldos en mora, huella de consulta).
La variable `Pago_atiempo` indica si el cliente cumplio (1) o no (0) con el pago.

**Objetivo de este notebook.** Cargar `Base_de_datos.csv`, verificar que se lee correctamente y hacer un primer
diagnostico de calidad: tipos de datos, nulos, duplicados, rangos y balance del target. Las decisiones de limpieza
se toman en `comprension_eda.ipynb` y se implementan en `ft_engineering.py`.

## 1. Setup e importacion de librerias

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)  # solo avisos de deprecacion de pandas/seaborn
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def buscar_raiz(nombre="Base_de_datos.csv"):
    """Sube por el arbol de carpetas hasta encontrar el dataset (permite correr el notebook desde cualquier cwd)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / nombre).exists():
            return carpeta
    raise FileNotFoundError(f"No se encontro {nombre} subiendo desde {actual}")


RAIZ = buscar_raiz()
RUTA_DATOS = RAIZ / "Base_de_datos.csv"
print("Raiz del proyecto:", RAIZ)

Raiz del proyecto: E:\pi-m5-riesgo-crediticio-CristianAtrio


## 2. Carga del dataset

El archivo original llego en Excel; se convirtio a CSV para que sea liviano y versionable. `fecha_prestamo` se parsea como fecha desde la carga.

In [2]:
df = pd.read_csv(RUTA_DATOS, parse_dates=["fecha_prestamo"])
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
df.head()

Filas: 10,763  |  Columnas: 23


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,"3,692,160.00",10,42,Independiente,8000000,2500000,341296,88.77,695.00,10,5,0.00,"51,258.00","51,258.00",0.00,5,0,0,"908,526.00",Estable,1
1,4,2025-04-22 09:47:35,"840,000.00",6,60,Empleado,3000000,2000000,124876,95.23,789.00,3,1,0.00,"8,673.00","8,673.00",0.00,0,0,2,"939,017.00",Creciente,1
2,9,2026-01-08 12:22:40,"5,974,028.40",10,36,Independiente,4036000,829000,529554,47.61,740.00,4,5,0.00,"18,702.00","18,702.00",0.00,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,"1,671,240.00",6,48,Empleado,1524547,498000,252420,95.23,837.00,4,4,0.00,"15,782.00","15,782.00",0.00,3,0,0,"1,536,193.00",Creciente,1
4,9,2025-04-26 11:24:26,"2,781,636.00",11,44,Empleado,5000000,4000000,217037,95.23,771.00,4,6,0.00,"204,804.00","204,804.00",0.00,3,0,1,"933,473.00",Creciente,1


## 3. Diccionario de variables

| Variable | Descripcion (interpretada del nombre y los valores) | Tipo esperado |
|---|---|---|
| `tipo_credito` | Codigo del producto de credito (4 y 9 concentran el 99%) | categorica codificada como entero |
| `fecha_prestamo` | Fecha y hora de desembolso | fecha |
| `capital_prestado` | Monto desembolsado | numerica |
| `plazo_meses` | Plazo del credito en meses | numerica discreta |
| `edad_cliente` | Edad del solicitante | numerica |
| `tipo_laboral` | Empleado / Independiente | categorica |
| `salario_cliente` | Ingreso declarado por el cliente | numerica |
| `total_otros_prestamos` | Deuda declarada en otros prestamos | numerica |
| `cuota_pactada` | Cuota mensual del credito | numerica |
| `puntaje` | Score interno (ver alerta de fuga de informacion en el EDA) | numerica |
| `puntaje_datacredito` | Score de la central de riesgo (Datacredito) | numerica |
| `cant_creditosvigentes` | Cantidad de creditos vigentes en la central | numerica discreta |
| `huella_consulta` | Cantidad de consultas recientes a la central (busqueda de credito) | numerica discreta |
| `saldo_mora` | Saldo en mora del cliente en la central | numerica |
| `saldo_total` | Saldo total de deuda en la central | numerica |
| `saldo_principal` | Saldo de capital en la central | numerica |
| `saldo_mora_codeudor` | Saldo en mora como codeudor | numerica |
| `creditos_sectorFinanciero` | Creditos con bancos / financieras | numerica discreta |
| `creditos_sectorCooperativo` | Creditos con cooperativas | numerica discreta |
| `creditos_sectorReal` | Creditos con comercio / sector real | numerica discreta |
| `promedio_ingresos_datacredito` | Ingreso promedio estimado por la central | numerica |
| `tendencia_ingresos` | Creciente / Estable / Decreciente | categorica |
| `Pago_atiempo` | **Target**: 1 = pago a tiempo, 0 = no pago a tiempo | binaria |

## 4. Tipos de datos

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   tipo_credito                   10763 non-null  int64         
 1   fecha_prestamo                 10763 non-null  datetime64[us]
 2   capital_prestado               10763 non-null  float64       
 3   plazo_meses                    10763 non-null  int64         
 4   edad_cliente                   10763 non-null  int64         
 5   tipo_laboral                   10763 non-null  str           
 6   salario_cliente                10763 non-null  int64         
 7   total_otros_prestamos          10763 non-null  int64         
 8   cuota_pactada                  10763 non-null  int64         
 9   puntaje                        10763 non-null  float64       
 10  puntaje_datacredito            10757 non-null  float64       
 11  cant_creditosvigentes     

**Conclusion.** Los tipos son coherentes con el diccionario salvo dos casos: `tendencia_ingresos` se lee como
`object` pero deberia tener solo 3 categorias (veremos que trae valores numericos mezclados) y `tipo_credito`
es un entero que en realidad representa una categoria, no una cantidad. Ambos se tratan en el EDA.

## 5. Valores nulos

In [4]:
nulos = (
    df.isna().sum().to_frame("nulos")
    .assign(pct=lambda t: (t["nulos"] / len(df) * 100).round(2))
    .query("nulos > 0")
    .sort_values("nulos", ascending=False)
)
nulos

,nulos,pct
tendencia_ingresos,2932,27.24
promedio_ingresos_datacredito,2930,27.22
saldo_mora_codeudor,590,5.48
saldo_principal,405,3.76
saldo_mora,156,1.45
saldo_total,156,1.45
puntaje_datacredito,6,0.06


**Conclusion.** Hay nulos en 7 de 23 columnas y todos provienen de la central de riesgo, no del formulario del cliente:

- `promedio_ingresos_datacredito` y `tendencia_ingresos` faltan juntas en ~27% de los casos: la central no tiene
  estimacion de ingresos para ese cliente. Es un nulo *informativo* (cliente con poca historia crediticia), por lo que
  conviene imputar y ademas conservar un indicador de faltante.
- `saldo_mora`, `saldo_total`, `saldo_principal` y `saldo_mora_codeudor` faltan en 1,5% a 5,5% de los casos. Son saldos:
  la imputacion natural es 0 (sin deuda reportada) o la mediana, se decide en el EDA mirando la tasa de mora de esos grupos.
- `puntaje_datacredito` solo tiene 6 nulos.

## 6. Duplicados

In [5]:
print("Filas duplicadas completas:", df.duplicated().sum())
print("Filas duplicadas ignorando la fecha:", df.drop(columns="fecha_prestamo").duplicated().sum())

Filas duplicadas completas:

 0
Filas duplicadas ignorando la fecha: 0


**Conclusion.** No hay filas duplicadas, ni siquiera ignorando la fecha de desembolso. No se elimina nada.

## 7. Estadisticos descriptivos

In [6]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
tipo_credito,"10,763.00",5.41,4.00,4.00,4.00,9.00,68.00,2.34
fecha_prestamo,10763,2025-04-16 23:06:02.111121,2024-11-26 09:17:04,2025-01-20 17:33:07.500000,2025-03-27 16:23:12,2025-06-16 13:27:58,2026-04-26 18:43:52,NaN
capital_prestado,"10,763.00","2,434,315.00","360,000.00","1,224,831.00","1,921,920.00","3,084,840.00","41,444,152.80","1,909,642.76"
plazo_meses,"10,763.00",10.58,2.00,6.00,10.00,12.00,90.00,6.63
edad_cliente,"10,763.00",43.95,19.00,33.00,42.00,53.00,123.00,15.06
salario_cliente,"10,763.00","17,216,431.46",0.00,"2,000,000.00","3,000,000.00","4,875,808.00","22,000,000,000.00","355,476,717.60"
total_otros_prestamos,"10,763.00","6,238,869.65",0.00,"500,000.00","1,000,000.00","2,000,000.00","6,787,675,263.00","118,418,316.94"
cuota_pactada,"10,763.00","243,617.41","23,944.00","121,041.50","182,863.00","287,833.50","3,816,752.00","210,493.69"
puntaje,"10,763.00",91.17,-38.01,95.23,95.23,95.23,95.23,16.47
puntaje_datacredito,"10,757.00",780.79,-7.00,757.00,791.00,825.00,999.00,104.88


In [7]:
df.select_dtypes(exclude=["number", "datetime"]).describe().T

,count,unique,top,freq
tipo_laboral,10763,2,Empleado,6754
tendencia_ingresos,7831,46,Creciente,5294


**Conclusion.** El `describe` ya deja ver problemas de calidad que el EDA va a cuantificar:

- `edad_cliente` llega a 123 anios y `salario_cliente` a 22.000 millones: hay valores imposibles o mal cargados.
- `puntaje` tiene minimo negativo (-38) y percentiles 25/50/75 identicos (95,23): es casi una constante con una cola rara.
- `puntaje_datacredito` tiene un minimo de -7 y un maximo de 999 cuando el score de la central va de 150 a 950: hay valores fuera de rango por ambos extremos.
- `plazo_meses` tiene un maximo de 90 meses para un solo credito.
- `tendencia_ingresos` muestra 46 valores unicos cuando deberian ser 3.

## 8. Balance de la variable objetivo

In [8]:
balance = df["Pago_atiempo"].value_counts().to_frame("clientes")
balance["pct"] = (balance["clientes"] / len(df) * 100).round(2)
balance.index = balance.index.map({1: "1 = pago a tiempo", 0: "0 = no pago a tiempo (mora)"})
balance

,clientes,pct
Pago_atiempo,,
1 = pago a tiempo,10252,95.25
0 = no pago a tiempo (mora),511,4.75


**Conclusion.** El dataset esta fuertemente desbalanceado: solo el **4,75% (511 creditos)** no pagaron a tiempo.
Esto condiciona todo el proyecto:

- El *accuracy* es inutil como metrica (un modelo que siempre dice "paga" acierta el 95%).
- Vamos a evaluar con ROC-AUC, PR-AUC, recall y F1 de la clase minoritaria, y a usar `class_weight` / `scale_pos_weight`.
- Para el modelado definimos la clase positiva como **mora = 1 - Pago_atiempo**, que es lo que el negocio quiere detectar.

## 9. Chequeos rapidos de consistencia

In [9]:
chequeos = {
    "edad_cliente >= 100": (df["edad_cliente"] >= 100).sum(),
    "salario_cliente == 0": (df["salario_cliente"] == 0).sum(),
    "salario_cliente > 100 millones": (df["salario_cliente"] > 1e8).sum(),
    "salario_cliente > 1.000 millones": (df["salario_cliente"] > 1e9).sum(),
    "total_otros_prestamos > 100 millones": (df["total_otros_prestamos"] > 1e8).sum(),
    "puntaje < 0": (df["puntaje"] < 0).sum(),
    "puntaje_datacredito < 150": (df["puntaje_datacredito"] < 150).sum(),
    "puntaje_datacredito > 950": (df["puntaje_datacredito"] > 950).sum(),
    "tendencia_ingresos fuera de {Creciente, Estable, Decreciente}": (
        df["tendencia_ingresos"].notna() & ~df["tendencia_ingresos"].isin(["Creciente", "Estable", "Decreciente"])
    ).sum(),
    "tipo_credito con menos de 30 casos": df["tipo_credito"].map(df["tipo_credito"].value_counts()).lt(30).sum(),
    "plazo_meses > 48": (df["plazo_meses"] > 48).sum(),
}
pd.Series(chequeos, name="casos").to_frame()

,casos
edad_cliente >= 100,150
salario_cliente == 0,24
salario_cliente > 100 millones,74
salario_cliente > 1.000 millones,22
total_otros_prestamos > 100 millones,44
puntaje < 0,135
puntaje_datacredito < 150,147
puntaje_datacredito > 950,6
"tendencia_ingresos fuera de {Creciente, Estable, Decreciente}",58
tipo_credito con menos de 30 casos,24


In [10]:
print("Rango de fechas:", df["fecha_prestamo"].min().date(), "->", df["fecha_prestamo"].max().date())
print("Valores de tipo_credito:", df["tipo_credito"].value_counts().to_dict())
print("Valores de tipo_laboral:", df["tipo_laboral"].value_counts().to_dict())
print("Valores raros en tendencia_ingresos:")
print(df.loc[~df["tendencia_ingresos"].isin(["Creciente", "Estable", "Decreciente"]), "tendencia_ingresos"].value_counts(dropna=False).head(10))

Rango de fechas: 2024-11-26 -> 2026-04-26
Valores de tipo_credito: {4: 7747, 9: 2876, 10: 116, 6: 21, 7: 2, 68: 1}
Valores de tipo_laboral: {'Empleado': 6754, 'Independiente': 4009}
Valores raros en tendencia_ingresos:
tendencia_ingresos
NaN        2932
0             7
8315          6
1000000       4
9147          2
158042        1
3978          1
168750        1
-28589        1
-566272       1
Name: count, dtype: int64


## 10. Resumen de la carga

| Aspecto | Resultado |
|---|---|
| Registros | 10.763 creditos desembolsados entre nov-2024 y abr-2026 |
| Variables | 23 columnas: 20 numericas (12 enteras, incluido el target binario, y 8 decimales), 1 fecha y 2 de texto |
| Nulos | 7 columnas, todas de la central de riesgo; `promedio_ingresos_datacredito` / `tendencia_ingresos` ~27% |
| Duplicados | 0 |
| Target | 95,25% pago a tiempo vs 4,75% mora: **fuerte desbalance** |
| Calidad | 150 edades entre 121 y 123; 74 salarios > 100 M (22 de ellos > 1.000 M); 58 valores numericos en `tendencia_ingresos`; `puntaje_datacredito` fuera de rango: 147 < 150 y 6 > 950 |

**Proximo paso:** `comprension_eda.ipynb` profundiza en la distribucion de cada variable, su relacion con la mora y las
correlaciones entre ellas, y deja escritas las decisiones de limpieza e ingenieria de caracteristicas.